In [34]:
%pip install numpy
%pip install pandas


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [35]:
import numpy as np
import pandas as pd
from pathlib import Path

data_dir = Path('/Users/cathycroome/data/plaques')
T = pd.read_csv(data_dir / 'open-plaques-United-Kingdom-2025-12-14.csv')

In [ ]:
def cartesian_to_polar(x, y):
    """
    Convert Cartesian coordinates to polar coordinates.
    Theta is measured anti-clockwise from the positive x-axis, in the range [0, 2*pi).
    """
    r = np.sqrt(x**2 + y**2)
    t = np.arctan2(y, x)  # radians, range (-pi, pi]

    # if y >= 0: quadrant 1 or 2, angle is already correct
    # if y < 0:  quadrant 3 or 4, add 2*pi to wrap into [0, 2*pi)
    theta = np.where(y >= 0, t, 2 * np.pi + t)

    theta_deg = np.degrees(theta)

    return r, theta_deg

def manual_lat_and_long(location):
    match location:
        case 'Norwich':
            lat, long = 52.633696, 1.289063
        case 'Brighton':
            lat, long = 50.818918, -0.140827
        case 'Aberdeenshire':
            lat, long = 57.581714, -2.627400
        case 'Llangollen':
            lat, long = 52.970465, -3.170968
        case 'York':
            lat, long = 53.960403, -1.081140
        case 'Formby':
            lat, long = 53.558966, -3.075819
        case _:
            raise ValueError(f"Unknown location: {location}")

    return lat, long

def relative_location(lat,long, centre_lat = 53.4153, centre_long=-2.2127):
    """
    Return (x, y) offset of a point from a fixed centre point,
    in degrees of longitude/latitude (not true distance).
    """
    dx_deg = long - centre_long
    dy_deg = lat - centre_lat

    return dx_deg, dy_deg

def degrees_to_km(dx_deg, dy_deg, lat, km_per_degree = 111.0):
    """
    Convert a (dx_deg, dy_deg) offset into approximate (x_km, y_km),
    used to correct for longitude lines converging away from the equator.
    """
    lat_rad = np.radians(lat)
    x_km = dx_deg * km_per_degree * np.cos(lat_rad)
    y_km = dy_deg * km_per_degree
    
    return x_km, y_km

In [ ]:
# test with manual locations 
location = 'York'                               # choose test location
lat, long = manual_lat_and_long(location)       # set latitude and longitude for location
dx_deg, dy_deg = relative_location(lat, long)   # change in lat and long relative to set centre point
x, y = degrees_to_km(dx_deg, dy_deg, lat)       # convert to cartesian (approximate distances in km)
r, theta_deg = cartesian_to_polar(x ,y)         # convert to polar

print('x = ', x)
print ('y = ', y)
print('r = ', r)
print('theta_deg = ', theta_deg)

x =  73.89789344166381
y =  60.50643299999972
r =  95.50878016967316
theta =  0.6860902811111498
theta_deg =  39.31007747261311


In [38]:
# format data frame
subset = T[['id', 'lead_subject_name', 'latitude', 'longitude',  'area', 'address',  'colour']].copy()
subset = subset.dropna(subset=['latitude', 'longitude', 'lead_subject_name', 'colour'])

# calculate r and theta and append to data frame
dx_deg, dy_deg = relative_location(subset.latitude, subset.longitude)   # change in lat and long relative to set centre point
x, y = degrees_to_km(dx_deg, dy_deg, subset.latitude)                   # convert to cartesian (approximate distances in km)
r, theta, theta_deg = cartesian_to_polar(x,y)                           # convert to polar
subset['r'] = r
subset['theta_deg'] = theta_deg

subset.head(10)

,id,lead_subject_name,latitude,longitude,area,address,colour,r,theta_deg
0,9324,Charles Dickens,50.59723,-1.18632,"Ventnor, Isle of Wight","Shore Road, Bonchurch",blue,321.056592,283.017570
1,8973,Henry Maudslay,51.39432,0.52742,Chatham,"Chatham Historical Dockyard, The Old Surgery, ...",blue,293.835539,310.230695
2,9850,Kew Bridge Pumping Station,51.48904,-0.29049,London,"Kew Bridge Steam Museum, Green Dragon Lane, Br...",grey,251.728498,301.854965
5,1577,John Jaffray,52.45567,-1.91450,Birmingham,"Consultancy Suite, Priory Hospital, Priory Roa...",blue,108.411849,280.722578
10,10522,William Penn,51.45246,-0.96787,Reading,"RISC Global Cafe, London Street",brass,234.273185,291.564401
11,3492,John Mann,51.99000,-1.70223,Moreton-in-Marsh,Oxford Street,stone,162.010349,282.437358
12,43997,Ernest Shackleton,55.95626,-3.22113,Edinburgh,14 South Learmonth Gardens,blue,288.924039,102.526384
15,8941,Bessemer Converter,53.38938,-1.47186,Sheffield,"Kelham Island Industrial Museum, Alma Street",grey,49.126062,356.642491
18,10019,William Hale White,51.36560,-0.16473,London,"Honeywood Walk, Carshalton",brown,268.156549,301.956811
19,12573,Rowe Bros & Co,52.47583,-1.90936,Birmingham,"CBSO Centre, Berkley Street",blue,106.278736,281.126220
